# IOAI — 2026 Summer National Corrupt Codex (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2026/nyari-orszagos/feladatok/adatok/korrupt-kodex >/dev/null && git checkout -q
import shutil, glob
for f in glob.glob('haio/2026/nyari-orszagos/feladatok/adatok/korrupt-kodex/*'): shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Corrupt Codex — Label-Noise / Memorization Detection (Korrupt Kódex)

> **HAIO 2026 — Summer National Finals (ML)**

A fraud-detection net (12→256→256→1, SIREN `sin` activations) was trained on data where some labels were **corrupted**; the net **memorized** those flipped points. Given the trained weights (`net_weights.pt`), 50 labelled calibration rows (`calibration.csv`, last col = `is_memorized`), and 5000 unlabelled test rows (`test.csv`), output a **score per test row** = likelihood it is a memorized/corrupted sample. Score = **ROC-AUC** vs the hidden truth. **Submit** `submission.csv` (`id,score`).

Baseline: memorized points sit near the decision boundary → use **logit uncertainty** (`-|logit|`), orientation calibrated on the 50 labelled rows. (Improve with perturbation flip-rate / input-gradient / activation-probe — see the reference solution.)

In [ ]:
import numpy as np, pandas as pd, torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
device='cuda' if torch.cuda.is_available() else 'cpu'
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(12, 256); self.fc2 = nn.Linear(256, 256); self.fc3 = nn.Linear(256, 1)
    def forward(self, x):
        x = torch.sin(30 * self.fc1(x)); x = torch.sin(30 * self.fc2(x)); return self.fc3(x)
model=Net().to(device)
model.load_state_dict(torch.load('net_weights.pt', map_location=device)); model.eval()
cal=pd.read_csv('calibration.csv'); test=pd.read_csv('test.csv')
Xcal=torch.tensor(cal.iloc[:, :12].values, dtype=torch.float32, device=device)
ycal=cal.iloc[:, 12].values
Xte=torch.tensor(test[[f'f{i}' for i in range(12)]].values, dtype=torch.float32, device=device)
len(cal), len(test)

## Logit-uncertainty signal (orientation calibrated on the 50 labelled rows)

In [ ]:
with torch.no_grad():
    lcal=model(Xcal).squeeze(1).cpu().numpy()
    lte=model(Xte).squeeze(1).cpu().numpy()
sig_cal=-np.abs(lcal); sig_te=-np.abs(lte)
auc=roc_auc_score(ycal, sig_cal)
if auc<0.5:
    sig_cal=-sig_cal; sig_te=-sig_te; auc=1-auc
print(f'calibration AUC (-|logit|): {auc:.4f}')
pd.DataFrame({'id':range(len(test)),'score':sig_te}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(test))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)